In [ ]:
import pandas as pd
from pathlib import Path

# Paths (adjust if needed)
ROOT = Path(__file__).parent if '__file__' in globals() else Path('.')
INPUT_DIR = ROOT / 'csvs'
OUTPUT_DIR = ROOT / 'corrected_data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = ['kse100_daily_data.csv', 'kse30_daily_data.csv']

In [ ]:
def canonical_company_map(df):
    # Trim whitespace from COMPANY and SYMBOL
    df = df.copy()
    if 'COMPANY' in df.columns:
        df['COMPANY'] = df['COMPANY'].astype(str).str.strip()
    if 'SYMBOL' in df.columns:
        df['SYMBOL'] = df['SYMBOL'].astype(str).str.strip()

    # For each SYMBOL pick the most frequent non-empty COMPANY (mode), else first non-null
    mapping = {}
    for sym, grp in df.groupby('SYMBOL', dropna=False):
        # get trimmed non-empty values
        vals = grp['COMPANY'].dropna().astype(str).str.strip()
        vals = vals[vals != '']
        if len(vals) == 0:
            mapping[sym] = ''
            continue
        # use mode if available
        try:
            m = vals.mode()
            if len(m) > 0:
                mapping[sym] = m.iloc[0]
            else:
                mapping[sym] = vals.iloc[0]
        except Exception:
            mapping[sym] = vals.iloc[0]
    return mapping

def coerce_numeric(series):
    s = series.astype(str).str.replace(',', '').str.replace('%', '').str.strip()
    return pd.to_numeric(s, errors='coerce')

def clean_df(df):
    df = df.copy()

    # Normalize SYMBOL and COMPANY
    if 'SYMBOL' in df.columns:
        df['SYMBOL'] = df['SYMBOL'].astype(str).str.strip()
    if 'COMPANY' in df.columns:
        df['COMPANY'] = df['COMPANY'].astype(str).str.strip()

    # Resolve volume column ambiguity: prefer 'VOLUME', fallback to 'Volume'
    if 'VOLUME' in df.columns and 'Volume' in df.columns:
        df['VOLUME'] = df['VOLUME'].combine_first(df['Volume'])
        try:
            df.drop(columns=['Volume'], inplace=True)
        except Exception:
            pass
    elif 'Volume' in df.columns and 'VOLUME' not in df.columns:
        df.rename(columns={'Volume': 'VOLUME'}, inplace=True)

    # Drop ISIN if present
    if 'ISIN' in df.columns:
        df = df.drop(columns=['ISIN'])

    # Canonicalize COMPANY by SYMBOL
    if 'SYMBOL' in df.columns and 'COMPANY' in df.columns:
        cmap = canonical_company_map(df)
        df['COMPANY'] = df['SYMBOL'].map(cmap).fillna('')

    # Numeric columns to coerce
    numeric_cols = ['PRICE','IDX WT %','FF BASED SHARES','FF BASED MCAP','ORD SHARES','ORD SHARES MCAP','VOLUME']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = coerce_numeric(df[col])

    return df

In [ ]:
# Process files and save cleaned versions
for fname in FILES:
    in_path = INPUT_DIR / fname
    if not in_path.exists():
        print(f'File not found: {in_path} — skipping')
        continue
    print('Reading', in_path)
    df = pd.read_csv(in_path)
    cleaned = clean_df(df)
    out_name = 'cleaned_' + fname
    out_path = OUTPUT_DIR / out_name
    cleaned.to_csv(out_path, index=False)
    print(f'Wrote cleaned file: {out_path} (rows={len(cleaned)})')
    display(cleaned.head())
    print('Dtypes:')
    print(cleaned.dtypes)

Notes:
- The canonical name selection uses the most frequent (mode) trimmed COMPANY string per SYMBOL. If you prefer a different rule (first seen, authoritative mapping file, manual overrides), modify the `canonical_company_map` function.
- Numeric coercion uses `errors='coerce'` — invalid values become `NaN`. Consider filling those if desired.
- The notebook writes outputs into `corrected_data/cleaned_kse100_daily_data.csv` and `cleaned_kse30_daily_data.csv`.